# MLOps Training 2026/2027 — Task 2
## Notebook 2: Create the Labels
In this notebook, I will create the target label for the delivery prediction problem.

An order will be labeled as late if the actual delivery date is after the estimated delivery date. I will check the label on a few orders, look at the class distribution, and then save the labeled table for the next notebook.

## 1. Load the ML Table
The labeled dataset will be built from the ML table created in Notebook 1. I will load that saved artifact first and check its shape before creating the target label.

In [1]:
import pandas as pd

ml_table = pd.read_csv("artifacts/ml_table.csv")

print("Rows:", ml_table.shape[0])
print("Columns:", ml_table.shape[1])

ml_table.head()

Rows: 99441
Columns: 29


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_count,total_payment_value,max_installments,payment_type_count,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude,distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.0,38.71,1.0,2.0,9350.0,maua,SP,-23.680729,-46.444238,18.576110
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,141.46,1.0,1.0,31570.0,belo horizonte,SP,-19.807681,-43.980427,851.495069
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,179.12,3.0,1.0,14840.0,guariba,SP,-21.363502,-48.229601,514.410666
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,72.20,1.0,1.0,31842.0,belo horizonte,MG,-19.837682,-43.924053,1822.226336
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,28.62,1.0,1.0,8752.0,mogi das cruzes,SP,-23.543395,-46.262086,29.676625


## 2. Prepare the Delivery Dates
The target label is based on the actual delivery date and the estimated delivery date. Before creating the label, I will convert both columns to datetime and check for missing values.

In [2]:
ml_table["order_delivered_customer_date"] = pd.to_datetime(
    ml_table["order_delivered_customer_date"]
)

ml_table["order_estimated_delivery_date"] = pd.to_datetime(
    ml_table["order_estimated_delivery_date"]
)

print(
    "Missing actual delivery dates:",
    ml_table["order_delivered_customer_date"].isna().sum()
)

print(
    "Missing estimated delivery dates:",
    ml_table["order_estimated_delivery_date"].isna().sum()
)

Missing actual delivery dates: 2965
Missing estimated delivery dates: 0


There are 2,965 orders without an actual delivery date, while all orders have an estimated delivery date.

Since the label depends on comparing the actual and estimated delivery dates, orders without an actual delivery date cannot be labeled as late or on time. I will exclude these rows from the labeled dataset.

In [3]:
labeled_table = ml_table[
    ml_table["order_delivered_customer_date"].notna()
].copy()

print("Original rows:", len(ml_table))
print("Rows available for labeling:", len(labeled_table))
print("Rows excluded:", len(ml_table) - len(labeled_table))

Original rows: 99441
Rows available for labeling: 96476
Rows excluded: 2965


## 3. Create the Delivery Label
An order is considered late when the actual delivery date is after the estimated delivery date.

The label will be:

- `1` = Late
- `0` = On time

In [4]:
labeled_table["late_delivery"] = (
    labeled_table["order_delivered_customer_date"].dt.date
    > labeled_table["order_estimated_delivery_date"].dt.date
).astype(int)

labeled_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "late_delivery"
    ]
].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,0
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,0


In [9]:
print("Rows:", labeled_table.shape[0])
print("Columns:", labeled_table.shape[1])

Rows: 96476
Columns: 30


### 3.1 Check the Label
Before using the label, I will check a few late and on-time orders to make sure the dates match the assigned label.

In [5]:
late_examples = labeled_table[
    labeled_table["late_delivery"] == 1
][
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "late_delivery"
    ]
].head(5)

on_time_examples = labeled_table[
    labeled_table["late_delivery"] == 0
][
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "late_delivery"
    ]
].head(5)

print("Late orders:")
display(late_examples)

print("On-time orders:")
display(on_time_examples)

Late orders:


,order_id,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-10-09 22:23:46,2017-09-28,1
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-03-21 22:03:54,2018-03-12,1
41,6ea2f835b4556291ffdc53fa0b3b95e8,2017-12-28 18:59:23,2017-12-21,1
57,66e4624ae69e7dc89bd50222b59f581f,2018-04-03 13:28:46,2018-04-02,1
58,a685d016c8a26f71a0bb67821070e398,2017-04-06 13:37:16,2017-03-30,1


On-time orders:


,order_id,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0


## 4. Class Distribution
Now I will check how many orders are late and how many are on time. This will show whether the target classes are balanced.

In [6]:
class_counts = labeled_table["late_delivery"].value_counts().sort_index()

class_percentages = (
    labeled_table["late_delivery"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

class_distribution = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages.round(2)
})

class_distribution.index = ["On time (0)", "Late (1)"]

class_distribution

,count,percentage
On time (0),89941,93.23
Late (1),6535,6.77


### 4.1 Class Imbalance
The target is imbalanced. About 93.23% of the orders are on time, while 6.77% are late.

Late deliveries are a much smaller class, so accuracy alone may not be a good measure of model performance. This class imbalance will be considered later when training and evaluating the model.

## 5. Save the Labeled Table
The labeled table will be saved as an artifact for Notebook 3. It contains only orders with an actual delivery date and includes the `late_delivery` target.

In [7]:
labeled_table.to_csv(
    "artifacts/labeled_table.csv",
    index=False
)

print("Labeled table saved successfully!")

Labeled table saved successfully!


## 6. Final Check
Before finishing this notebook, I will confirm that the saved labeled table contains the expected number of rows and that the target label has no missing values.

In [8]:
saved_labeled_table = pd.read_csv("artifacts/labeled_table.csv")

print("Saved rows:", len(saved_labeled_table))
print("Unique order IDs:", saved_labeled_table["order_id"].nunique())
print("Duplicate order IDs:", saved_labeled_table["order_id"].duplicated().sum())
print("Missing labels:", saved_labeled_table["late_delivery"].isna().sum())

Saved rows: 96476
Unique order IDs: 96476
Duplicate order IDs: 0
Missing labels: 0


## 7. Conclusion
The delivery label was created by comparing the actual delivery date with the estimated delivery date. Orders without an actual delivery date were excluded because they could not be labeled.

The final labeled dataset contains 96,476 orders. About 93.23% are on time and 6.77% are late, showing that the target classes are imbalanced.

The labeled table has been saved as `labeled_table.csv` and will be used in Notebook 3.